# Project 4 — Temporal Decision Aggregation

Formalizes the "repeated classification" idea with real prior art: Englehart & Hudgins introduced majority voting as a post-processing step over consecutive classifier decisions, and overlapped windowing (as used in [Project 1](01_emg_gesture_classification.ipynb)) is what makes majority voting tractable in the first place, since it raises the decision rate.

**Dataset**: Ninapro DB2 again, reusing the Project 1 pipeline directly (this notebook imports the data loading / preprocessing / windowing / feature / CNN code from `01_emg_gesture_classification.ipynb` rather than redefining it).

## Two experiments — the contrast between them is the actual finding

- **(a) Consecutive-window voting**: slide a 200 ms window at 100 ms increments through a single 5 s repetition, majority-vote over the last *k* decisions. Overlapping windows share the same underlying contraction, so their errors are strongly correlated — the empirical gain should fall well short of the independent-error binomial prediction.
- **(b) Across-repetition voting**: take *k* separate repetitions of the same gesture, one decision each, vote. Errors here are much closer to independent, so gains should approach the binomial ceiling.

## Deliverables
1. Empirical vs. theoretical (binomial-ceiling) accuracy for k = 1, 3, 5, 7, 9 in both experiments.
2. Error-correlation quantification via the pairwise Q-statistic and the correlation of error-indicator vectors.
3. Latency pricing: k votes at a 100 ms decision increment costs (k−1)·100 ms; plot accuracy vs. total latency with the 300 ms usability line.
4. Wolpaw information transfer rate (ITR, bits/min) vs. k — the actual engineering answer, trading accuracy against decision rate.
5. Majority voting vs. confidence-weighted voting (sum of softmax probs), EWMA over the probability stream, and a rejection threshold that emits nothing below a confidence floor.


## 1. Setup

Reuses Project 1's data/feature/model code. In Colab, run `01_emg_gesture_classification.ipynb` first (or `%run` it) so `CFG`, `prepare_subject`, `make_windows`, `EMG1DCNN`, etc. are in scope.

In [ ]:
# Colab setup
# !pip install -q scipy scikit-learn torch numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import comb
from itertools import combinations

import torch

RNG_SEED = 0
np.random.seed(RNG_SEED)

# Pull in Project 1's pipeline. In Colab: %run "01_emg_gesture_classification.ipynb"
# Locally, either run that notebook first in the same kernel, or refactor its cells
# into a shared `emg_pipeline.py` and `import` it here.
try:
    CFG
    prepare_subject
    make_windows
    fit_cnn
    predict_cnn
    EMG1DCNN
except NameError as e:
    raise RuntimeError(
        "Run 01_emg_gesture_classification.ipynb first (or %run it) so CFG, "
        "prepare_subject, make_windows, fit_cnn, predict_cnn, EMG1DCNN are defined."
    ) from e


## 2. Get a trained model + per-window probability stream

We need calibrated per-window softmax probabilities, not just argmax predictions, for the confidence-weighted variants below.

In [ ]:
@torch.no_grad()
def predict_proba_cnn(model, X_windows, batch_size=256):
    from torch.utils.data import DataLoader
    model.eval()
    loader = DataLoader(WindowDataset(X_windows, np.zeros(len(X_windows))), batch_size=batch_size)
    probs = []
    for xb, _ in loader:
        out = model(xb.to(DEVICE))
        probs.append(torch.softmax(out, dim=1).cpu().numpy())
    return np.concatenate(probs)


def get_repetition_stream(rec, cfg: Config, model, mu, sd, repetition_id: int):
    """Windows + probability stream for ONE repetition of ONE subject, in time order."""
    emg, y, rep = rec["emg"], rec["restimulus"], rec["rerepetition"]
    mask = rep == repetition_id
    Xw, yw, repw = make_windows(emg[mask], y[mask], rep[mask], cfg)
    Xw_norm = (Xw - mu) / sd
    probs = predict_proba_cnn(model, Xw_norm)
    preds = probs.argmax(axis=1)
    return {"X": Xw, "y": yw, "probs": probs, "preds": preds}


In [ ]:
files = list_subject_files(CFG)
if files:
    rec = prepare_subject(files[0], CFG)
    n_classes = CFG.n_classes_subset + 1

    Xw_all, yw_all, repw_all = make_windows(rec["emg"], rec["restimulus"], rec["rerepetition"], CFG)
    tr_mask, te_mask = split_within_subject(yw_all, repw_all, CFG)

    mu = Xw_all[tr_mask].mean(axis=(0, 1), keepdims=True)
    sd = Xw_all[tr_mask].std(axis=(0, 1), keepdims=True) + 1e-8
    Xtr = (Xw_all[tr_mask] - mu) / sd

    print("training base CNN for the voting experiments...")
    base_model = fit_cnn(Xtr, yw_all[tr_mask], n_classes, CFG, epochs=15)
else:
    print("No Ninapro DB2 files found — populate CFG.data_dir, then re-run.")


## 3. Voting rules

In [ ]:
def majority_vote(preds_window):
    vals, counts = np.unique(preds_window, return_counts=True)
    return vals[np.argmax(counts)]


def confidence_weighted_vote(probs_window):
    """Sum softmax probabilities across the k windows, argmax the sum."""
    return probs_window.sum(axis=0).argmax()


def ewma_vote(probs_window, alpha=0.3):
    """Exponentially weighted moving average over the probability stream, most recent last."""
    ewma = probs_window[0]
    for p in probs_window[1:]:
        ewma = alpha * p + (1 - alpha) * ewma
    return ewma.argmax()


def rejection_vote(probs_window, floor=0.5):
    """Majority vote, but abstain (-1) if the winning class's mean confidence is below floor."""
    mean_probs = probs_window.mean(axis=0)
    top = mean_probs.argmax()
    if mean_probs[top] < floor:
        return -1
    return top


def sliding_k_windows(n, k):
    """Yield index arrays for consecutive windows of length k (experiment a)."""
    for start in range(0, n - k + 1):
        yield np.arange(start, start + k)


## 4. Experiment (a): consecutive-window voting within one repetition

In [ ]:
def run_consecutive_experiment(stream, ks=(1, 3, 5, 7, 9)):
    preds, probs, y = stream["preds"], stream["probs"], stream["y"]
    n = len(preds)
    rows = []
    for k in ks:
        idx_groups = list(sliding_k_windows(n, k))
        if not idx_groups:
            continue
        for idx in idx_groups:
            true = y[idx[-1]]  # label at the most recent window in the group
            maj = majority_vote(preds[idx])
            cw = confidence_weighted_vote(probs[idx])
            ew = ewma_vote(probs[idx])
            rej = rejection_vote(probs[idx])
            rows.append({"k": k, "true": true, "majority": maj, "conf_weighted": cw,
                         "ewma": ew, "rejection": rej})
    return pd.DataFrame(rows)


if files:
    consecutive_rows = []
    for subj_rec in [rec]:  # extend to more subjects for a fuller sweep
        for repetition_id in np.unique(subj_rec["rerepetition"]):
            stream = get_repetition_stream(subj_rec, CFG, base_model, mu, sd, repetition_id)
            if len(stream["y"]) < 9:
                continue
            df = run_consecutive_experiment(stream)
            df["subject"] = subj_rec["subject"]
            df["repetition"] = repetition_id
            consecutive_rows.append(df)
    consecutive_df = pd.concat(consecutive_rows, ignore_index=True) if consecutive_rows else pd.DataFrame()
    display(consecutive_df.head())


## 5. Experiment (b): across-repetition voting

One decision per repetition (majority window-vote within the repetition, or just the whole-repetition CNN prediction), then vote across *k* repetitions of the same gesture.

In [ ]:
def run_across_repetition_experiment(subj_rec, cfg, model, mu, sd, ks=(1, 3, 5, 7, 9)):
    reps = np.unique(subj_rec["rerepetition"])
    # one aggregate decision + averaged prob vector per (repetition, true class)
    per_rep = []
    for r in reps:
        stream = get_repetition_stream(subj_rec, cfg, model, mu, sd, r)
        if len(stream["y"]) == 0:
            continue
        true_class = int(pd.Series(stream["y"]).mode()[0])
        rep_pred = majority_vote(stream["preds"])
        rep_prob = stream["probs"].mean(axis=0)
        per_rep.append({"true": true_class, "pred": rep_pred, "prob": rep_prob})

    rows = []
    by_class = {}
    for r in per_rep:
        by_class.setdefault(r["true"], []).append(r)

    for true_class, entries in by_class.items():
        n_entries = len(entries)
        for k in ks:
            if k > n_entries:
                continue
            for combo in combinations(range(n_entries), k):
                sel = [entries[i] for i in combo]
                preds_k = np.array([e["pred"] for e in sel])
                probs_k = np.stack([e["prob"] for e in sel])
                rows.append({
                    "k": k, "true": true_class,
                    "majority": majority_vote(preds_k),
                    "conf_weighted": confidence_weighted_vote(probs_k),
                })
    return pd.DataFrame(rows)


if files:
    across_df = run_across_repetition_experiment(rec, CFG, base_model, mu, sd)
    display(across_df.head())


## 6. Empirical vs. theoretical (binomial ceiling)

For per-decision accuracy p and odd k, the independent-error prediction is  
$$P(\text{correct}) = \sum_{i > k/2} \binom{k}{i} p^i (1-p)^{k-i}$$

In [ ]:
def binomial_ceiling(p, k):
    thresh = k // 2  # need i > k/2, odd k
    total = 0.0
    for i in range(thresh + 1, k + 1):
        total += comb(k, i) * (p ** i) * ((1 - p) ** (k - i))
    return total


def summarize_experiment(df, base_p):
    rows = []
    for k, g in df.groupby("k"):
        emp_acc = (g["majority"] == g["true"]).mean()
        theo_acc = binomial_ceiling(base_p, k) if k % 2 == 1 else np.nan
        rows.append({"k": k, "empirical": emp_acc, "theoretical": theo_acc})
    return pd.DataFrame(rows).sort_values("k")


if files and len(consecutive_df) and len(across_df):
    k1 = consecutive_df.query("k == 1")
    base_p = float((k1["majority"] == k1["true"]).mean())

    consec_summary = summarize_experiment(consecutive_df, base_p)
    across_summary = summarize_experiment(across_df, base_p)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, summ, title in [(axes[0], consec_summary, "Consecutive-window (correlated errors)"),
                              (axes[1], across_summary, "Across-repetition (~independent errors)")]:
        ax.plot(summ["k"], summ["empirical"], "o-", label="empirical")
        ax.plot(summ["k"], summ["theoretical"], "s--", label="binomial ceiling")
        ax.set_xlabel("k (votes)")
        ax.set_title(title)
        ax.legend()
    axes[0].set_ylabel("accuracy")
    plt.tight_layout()
    plt.savefig("project4_binomial_ceiling.png", dpi=150)
    plt.show()


## 7. Quantifying error correlation: pairwise Q-statistic

In [ ]:
def q_statistic(correct_i, correct_j):
    """Yule's Q for two binary correctness vectors, paired per-sample."""
    n11 = np.sum(correct_i & correct_j)
    n10 = np.sum(correct_i & ~correct_j)
    n01 = np.sum(~correct_i & correct_j)
    n00 = np.sum(~correct_i & ~correct_j)
    denom = (n11 * n00 + n10 * n01)
    if denom == 0:
        return 0.0
    return (n11 * n00 - n10 * n01) / denom


def pairwise_q_for_stream(stream):
    """Q-statistic between window i and window i+1's correctness, averaged over the stream."""
    correct = (stream["preds"] == stream["y"])
    if len(correct) < 2:
        return np.nan
    return q_statistic(correct[:-1], correct[1:])


if files:
    qs = []
    for repetition_id in np.unique(rec["rerepetition"]):
        stream = get_repetition_stream(rec, CFG, base_model, mu, sd, repetition_id)
        if len(stream["y"]) >= 2:
            qs.append(pairwise_q_for_stream(stream))
    print(f"Mean adjacent-window Q-statistic (consecutive, correlated): {np.nanmean(qs):.3f}")
    print("Q closer to 1 => errors highly correlated => voting gain caps well below the binomial ceiling.")


## 8. Latency pricing and the 300 ms usability line

In [ ]:
def latency_ms(k, increment_ms=CFG.increment_ms if files else 100.0):
    return (k - 1) * increment_ms


if files:
    consec_summary["latency_ms"] = consec_summary["k"].apply(lambda k: latency_ms(k, CFG.increment_ms))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(consec_summary["latency_ms"], consec_summary["empirical"], "o-")
    ax.axvline(300, color="red", linestyle="--", label="300 ms usability budget")
    ax.set_xlabel("added latency (ms)")
    ax.set_ylabel("accuracy")
    ax.set_title("Accuracy vs. added voting latency")
    ax.legend()
    plt.tight_layout()
    plt.savefig("project4_latency_tradeoff.png", dpi=150)
    plt.show()


## 9. Wolpaw information transfer rate (ITR)

ITR (bits/trial) $= \log_2 N + p\log_2 p + (1-p)\log_2\!\big(\tfrac{1-p}{N-1}\big)$, converted to bits/min via the decision rate implied by k.

In [ ]:
def wolpaw_bits_per_trial(p, n_classes):
    if p <= 0 or p >= 1:
        p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / max(n_classes - 1, 1))


def itr_bits_per_min(p, n_classes, decision_interval_ms):
    bpt = wolpaw_bits_per_trial(p, n_classes)
    decisions_per_min = 60_000.0 / decision_interval_ms
    return max(bpt, 0.0) * decisions_per_min


if files:
    n_classes = CFG.n_classes_subset + 1
    consec_summary["decision_interval_ms"] = consec_summary["k"] * CFG.increment_ms
    consec_summary["itr_bpm"] = consec_summary.apply(
        lambda r: itr_bits_per_min(r["empirical"], n_classes, r["decision_interval_ms"]), axis=1)

    display(consec_summary)
    best_k = consec_summary.loc[consec_summary["itr_bpm"].idxmax(), "k"]
    print(f"k maximizing ITR: {best_k}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(consec_summary["k"], consec_summary["itr_bpm"], "o-")
    ax.axvline(best_k, color="green", linestyle="--", label=f"argmax k={best_k}")
    ax.set_xlabel("k (votes)")
    ax.set_ylabel("ITR (bits/min)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("project4_itr.png", dpi=150)
    plt.show()


## Notes

- This notebook demonstrates the full methodology on a **single subject** for speed; extend the loops in sections 4–5 over all subjects in `files` for a publishable sweep.
- The consecutive-vs-across-repetition gap (section 6 plot) is the headline finding: consecutive-window voting should visibly undershoot the binomial ceiling while across-repetition voting tracks it closely.
- The trained `base_model` here (same architecture as Project 1's CNN) is a natural handoff into [Project 5](03_quantization_efficiency.ipynb) — save it with `torch.save(base_model.state_dict(), "emg_cnn.pt")`.
